In [1]:
import os
import json
import urllib.request
import pandas as pd

# Descargar train-v2.0.json automáticamente si no existe localmente
json_path = "inputs/train-v2.0.json" if os.path.exists("inputs/train-v2.0.json") else "train-v2.0.json"
if not os.path.exists(json_path):
    url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json"
    print(f"Descargando {url}...")
    os.makedirs("inputs", exist_ok=True)
    urllib.request.urlretrieve(url, "inputs/train-v2.0.json")
    json_path = "inputs/train-v2.0.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Aplanar la estructura anidada de SQuAD 2.0 (data -> paragraphs -> qas)
rows = []
for article in data["data"]:
    title = article["title"]
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            rows.append({
                "title": title,
                "context": context,
                "question": qa["question"],
                "id": qa["id"],
                "is_impossible": qa.get("is_impossible", False),
                "answers": qa["answers"],
            })

df = pd.DataFrame(rows)
print(f"Total registros cargados: {len(df)}")
df.head()


Total registros cargados: 130319


In [2]:
df_por_contexto = (
    df.groupby("context")["question"]
    .apply(list)
    .reset_index(name="questions")
)
df_por_contexto.head()


In [3]:
df_por_contexto["num_questions"] = df_por_contexto["questions"].apply(len)
df_por_contexto["question_lens"] = df_por_contexto["questions"].apply(
    lambda qs: [len(q.split()) for q in qs]
)
df_por_contexto["avg_question_len"] = df_por_contexto["question_lens"].apply(
    lambda lens: sum(lens) / len(lens) if len(lens) > 0 else 0
)
df_por_contexto.head()


In [4]:
os.makedirs("outputs/processed", exist_ok=True)
df.to_csv("resultadoBase.csv", index=False)
df.to_csv("outputs/processed/resultadoBase.csv", index=False)
df_por_contexto.to_csv("outputs/processed/resultadoPorContexto.csv", index=False)
print("Archivos exportados exitosamente:")
print(" - resultadoBase.csv")
print(" - outputs/processed/resultadoBase.csv")
print(" - outputs/processed/resultadoPorContexto.csv")


Archivos exportados exitosamente:
 - resultadoBase.csv
 - outputs/processed/resultadoBase.csv
 - outputs/processed/resultadoPorContexto.csv
